In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [2]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_sft.json")

In [3]:
def apply_chat_template(user_prompt: str):
    system_prompt = "You are a helpful assistant."
    user_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n"
    return user_text

In [4]:
cfg = NSConfig()

device = "cuda:1"

cfg.vocab_size += 2
model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/sft_model_epoch_1.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32002, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32002, bias=False)
)

In [5]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("write a python sort function")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [6]:
input_tensor 

tensor([[32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 30370,   482,   261, 11202, 32001,
           224,   202,     1,     1],
        [32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 18617,   261, 14079, 13253,  3486,
          3054, 32001,   224,   202]], device='cuda:1')

In [7]:
attn_mask_tensor

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:1')

In [8]:
# no kv-cache + atten_mask

max_output = 500

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_tensor, attention_mask=attn_mask_tensor)
        # logits = model.forward(input_tensor, attention_mask=None)

        logits = logits[:, -1, :]

        # 概率采样
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
ged from the abuse of space-time during a great tragedy, some people think it's somehow inherently twisted or intended.

**Dang**

In a comedy, the characters must prove themselves to be paranoid or, worse, even intelligent. They can emote in a way that mediates the listener's emotions or develop a heightened sense of epistolary catharsis.

It's another possible angle: something about the extraterrestrial's website.

There is no point in trying to explain its existence or abductors' intent. Just how are they making our lives-ing.

**: The ultimate goal is not just to be perfect but to be spiritually enlightened. It's to be endlessly interesting and relatable, to allow for exploration of the world beyond space, to pay homage to the essence of space, and to help humanity outsmart its encroachment.

**: Even if the universe suddenly becomes the headquarters of our paranormal realm, it's not yet capable of solving the problem. If 

In [10]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("Where is the captail of the United States?")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [11]:
# kv-cache + atten_mask

max_output = 200


with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_tensor, use_cache=True, attention_mask=attn_mask_tensor)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)
        # NOTE: 使用kvcache时，只传入new token
        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True, attention_mask=attn_mask_tensor)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
- a motivational comedy that jumps right into the eye of the punter who's trying to tackle a tough task.

In this particular cartoon, it's not a one-way ticket to a delicious pie or a desert meal. In this cartoon, we see reality TV host Cary-Fashion, who ascends to the top of the Doomsday Hosts.

Here's the story:

Cary-Fashion: So, let's catch up...

Cary-Fashion: That's when it becomes clear. The Dunwich Horror Show is more than just a harmless bunch of scary comedians out in the early '60s looking for a shot at another TV show.

Cary-Fashion: Gosh, I'm all for big ideas with funny little snitches. But I've been living in the shadows for five years now, and I've only ever heard about this ridiculous idea.

Cary-F
----------------------------
 system
You are a helpful assistant. 
 user
Where is the captail of the United States? 
 assistant
The captail of the United States is the speediest device on the planet, but one aspect 